# Lifting ablation: does the lifting choice confound the GNN-vs-TNN comparison?

Reads `lifting_ablation_results.json` (written by `run_lifting_ablation.py`) and produces the two figures described in `README.md`. **This notebook only reads results and plots — it never trains a model.** Re-running it top to bottom should take seconds.

See `README.md` in this folder for the scientific question (H1, H2), the experimental design, and the finding.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd().resolve()
_IS_ANALYSIS_DIR = (
    (_HERE / "lifting_ablation_results.json").exists()
    or (_HERE / "run_lifting_ablation.py").exists()
)
_ANALYSIS_DIR = _HERE if _IS_ANALYSIS_DIR else _HERE.parent
_CHALLENGE_DIR = _ANALYSIS_DIR.parent.parent
for _p in (_CHALLENGE_DIR,):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Import the challenge `utils.py` (not `topobench.utils`) *before* any
# `topobench` import — see the note in `run_lifting_ablation.py` for why.
from utils import (
    AVG_DEGREE_LEVELS,
    HOMOPHILY_LEVELS,
    POWER_LAW_EXPONENT_LEVELS,
    apply_publication_matplotlib_style,
)

RESULTS_PATH = _ANALYSIS_DIR / "lifting_ablation_results.json"
FIGURES_DIR = _ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

apply_publication_matplotlib_style()

In [ ]:
with RESULTS_PATH.open(encoding="utf-8") as f:
    all_records = json.load(f)

ok_records = [r for r in all_records if r.get("status") == "ok"]
failed_records = [r for r in all_records if r.get("status") != "ok"]

print(f"{len(all_records)} total record(s): {len(ok_records)} ok, {len(failed_records)} failed.")
if failed_records:
    print("Failed runs:")
    for r in failed_records:
        print(f"  - {r.get('arm')}/{r.get('cell_key')}/s{r.get('seed')}: {r.get('error')}")

## Design constants

Cell order on the x-axis is fixed to `h_lo/pl_lo, h_lo/pl_hi, h_hi/pl_lo, h_hi/pl_hi` (`avg_degree` is held at `d_lo` throughout, see `README.md`).

In [ ]:
CELL_ORDER = [
    "h_lo__d_lo__pl_lo",
    "h_lo__d_lo__pl_hi",
    "h_hi__d_lo__pl_lo",
    "h_hi__d_lo__pl_hi",
]
CELL_LABELS = {
    "h_lo__d_lo__pl_lo": "h_lo / pl_lo",
    "h_lo__d_lo__pl_hi": "h_lo / pl_hi",
    "h_hi__d_lo__pl_lo": "h_hi / pl_lo",
    "h_hi__d_lo__pl_hi": "h_hi / pl_hi",
}

ARM_ORDER = ["khop1", "khop2", "knn3", "gcn"]
ARM_LABELS = {
    "khop1": "DPHGNN + khop (k=1)",
    "khop2": "DPHGNN + khop (k=2)",
    "knn3": "DPHGNN + knn (k=3)",
    "gcn": "GCN (no lifting)",
}
ARM_COLORS = {
    "khop1": "#1b9e77",
    "khop2": "#d95f02",
    "knn3": "#7570b3",
}

seeds_present = sorted({int(r["seed"]) for r in ok_records})
n_seeds = len(seeds_present)
print(f"Seeds present: {seeds_present}")

if n_seeds <= 1:
    print(
        "\n*** Phase 1 uses a single seed; differences within ~2 "
        "accuracy points are not interpretable. ***"
    )

In [ ]:
def mean_accuracy(arm, cell_key):
    """Return the mean test accuracy for (arm, cell_key) over available seeds, or NaN."""
    vals = [
        r["test_accuracy"]
        for r in ok_records
        if r["arm"] == arm and r["cell_key"] == cell_key
        and r["test_accuracy"] is not None
    ]
    return float(np.mean(vals)) if vals else float("nan")


acc_table = {
    arm: [mean_accuracy(arm, cell) for cell in CELL_ORDER] for arm in ARM_ORDER
}
for arm in ARM_ORDER:
    print(ARM_LABELS[arm], [f"{v:.3f}" if np.isfinite(v) else "NA" for v in acc_table[arm]])

## Figure 1 — lifting arms by structural regime

One line per DPHGNN lifting arm, GCN as a dashed grey reference line. Vertical dotted markers flag any x position where the rank order changes from the previous position — a rank inversion is the formal signature of H2.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(CELL_ORDER))

for arm in ["khop1", "khop2", "knn3"]:
    y = acc_table[arm]
    ax.plot(
        x, y, marker="o", linewidth=2.2, markersize=7,
        color=ARM_COLORS[arm], label=ARM_LABELS[arm],
    )

ax.plot(
    x, acc_table["gcn"], marker="s", linewidth=2.0, markersize=6,
    color="#666666", linestyle="--", label="GCN (no lifting)",
)

all_y = [v for arm in ARM_ORDER for v in acc_table[arm] if np.isfinite(v)]
y_lo, y_hi = (min(all_y), max(all_y)) if all_y else (0.0, 1.0)
y_span = max(y_hi - y_lo, 1e-6)
ax.set_ylim(y_lo - 0.08 * y_span, y_hi + 0.22 * y_span)
annotation_y = y_hi + 0.10 * y_span

# Rank-inversion annotations: compare the DPHGNN-arm ranking at each
# cell against the previous cell.
prev_rank = None
for i, cell in enumerate(CELL_ORDER):
    pairs = [(arm, acc_table[arm][i]) for arm in ["khop1", "khop2", "knn3"]]
    finite_pairs = [(a, v) for a, v in pairs if np.isfinite(v)]
    rank = tuple(a for a, _ in sorted(finite_pairs, key=lambda t: -t[1]))
    if prev_rank is not None and rank != prev_rank and rank and prev_rank:
        ax.axvline(i, color="crimson", linestyle=":", linewidth=1.6, alpha=0.85, zorder=0)
        ax.text(
            i, annotation_y, "rank\ninversion",
            ha="center", va="bottom", fontsize=9, color="crimson", fontweight="600",
        )
    if finite_pairs:
        prev_rank = rank

ax.set_xticks(x)
ax.set_xticklabels([CELL_LABELS[c] for c in CELL_ORDER])
ax.set_xlim(x[0] - 0.3, x[-1] + 0.3)
ax.set_xlabel("Grid cell (homophily / power-law), avg_degree = d_lo")
ax.set_ylabel("Test accuracy")
phase_tag = "Phase 1 (n=1 seed)" if n_seeds <= 1 else f"Phase 2 (n={n_seeds} seeds)"
ax.set_title(
    f"Lifting choice vs. structural regime — community detection, {phase_tag}",
    pad=14,
)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig1_lifting_by_regime.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 2 — rank table

4x4 table (cells x arms incl. GCN): rank 1 = best, accuracy in parentheses, colour-shaded by rank. Final row gives each arm's mean rank across the four cells.

In [ ]:
n_cells = len(CELL_ORDER)
n_arms = len(ARM_ORDER)
rank_matrix = np.full((n_cells, n_arms), np.nan)

for i, cell in enumerate(CELL_ORDER):
    accs = [acc_table[arm][i] for arm in ARM_ORDER]
    order = np.argsort([-a if np.isfinite(a) else np.inf for a in accs])
    ranks = np.empty(n_arms)
    r = 1
    for idx in order:
        if not np.isfinite(accs[idx]):
            ranks[idx] = np.nan
            continue
        ranks[idx] = r
        r += 1
    rank_matrix[i, :] = ranks

mean_rank = np.nanmean(rank_matrix, axis=0)

fig2, ax2 = plt.subplots(figsize=(8.5, 3.6))
display_matrix = np.vstack([rank_matrix, mean_rank[None, :]])
masked = np.ma.masked_invalid(display_matrix)
im = ax2.imshow(masked, cmap="RdYlGn_r", vmin=1, vmax=n_arms, aspect="auto")

ax2.set_xticks(range(n_arms))
ax2.set_xticklabels([ARM_LABELS[a] for a in ARM_ORDER], rotation=20, ha="right")
ax2.set_yticks(range(n_cells + 1))
ax2.set_yticklabels([CELL_LABELS[c] for c in CELL_ORDER] + ["Mean rank"])

for i in range(n_cells):
    for j in range(n_arms):
        rk, acc = rank_matrix[i, j], acc_table[ARM_ORDER[j]][i]
        txt = f"{int(rk)} ({acc:.3f})" if np.isfinite(rk) else "NA"
        ax2.text(j, i, txt, ha="center", va="center", fontsize=10, fontweight="600")
for j in range(n_arms):
    ax2.text(j, n_cells, f"{mean_rank[j]:.2f}", ha="center", va="center", fontsize=10, fontweight="700")

ax2.axhline(n_cells - 0.5, color="black", linewidth=1.5)
ax2.set_title("Rank per cell (1 = best), accuracy in parentheses")
fig2.colorbar(im, ax=ax2, label="Rank (1 = best)", shrink=0.85)
fig2.tight_layout()
fig2.savefig(FIGURES_DIR / "fig2_rank_table.png", dpi=300, bbox_inches="tight")
plt.show()

## Phase 2 statistics (only meaningful once `run_lifting_ablation.py --phase2` has produced 3 seeds per cell)

Two-way ANOVA (`arm` x `cell`, response = test accuracy), reporting partial eta^2 for `arm`, `cell`, and their interaction — the interaction term is the formal evidence for H2 (rank inversion). Uses `statsmodels` if available, otherwise computes the sums of squares manually (no new dependency).

In [ ]:
def bootstrap_ci(values, n_boot=2000, ci=0.95, rng=None):
    """Return a (mean, lo, hi) percentile bootstrap CI for `values`."""
    values = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if values.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = rng or np.random.default_rng(0)
    boots = rng.choice(values, size=(n_boot, values.size), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return float(values.mean()), float(lo), float(hi)


def manual_two_way_anova(df_rows):
    """Compute partial eta^2 for arm, cell, and arm:cell via manual sums of squares.

    `df_rows` is a list of (arm, cell, accuracy) tuples with >=1 replicate
    per (arm, cell) cell. Used only when `statsmodels` is unavailable.
    """
    arms = sorted({a for a, _, _ in df_rows})
    cells = sorted({c for _, c, _ in df_rows})
    grand_mean = np.mean([v for _, _, v in df_rows])

    ss_arm = 0.0
    for a in arms:
        vals = [v for aa, _, v in df_rows if aa == a]
        ss_arm += len(vals) * (np.mean(vals) - grand_mean) ** 2

    ss_cell = 0.0
    for c in cells:
        vals = [v for _, cc, v in df_rows if cc == c]
        ss_cell += len(vals) * (np.mean(vals) - grand_mean) ** 2

    ss_cells_combo = 0.0
    for a in arms:
        for c in cells:
            vals = [v for aa, cc, v in df_rows if aa == a and cc == c]
            if vals:
                ss_cells_combo += len(vals) * (np.mean(vals) - grand_mean) ** 2
    ss_interaction = ss_cells_combo - ss_arm - ss_cell

    ss_total = sum((v - grand_mean) ** 2 for _, _, v in df_rows)
    ss_error = ss_total - ss_cells_combo

    def partial_eta2(ss_effect):
        denom = ss_effect + ss_error
        return float(ss_effect / denom) if denom > 0 else float("nan")

    return {
        "partial_eta2_arm": partial_eta2(ss_arm),
        "partial_eta2_cell": partial_eta2(ss_cell),
        "partial_eta2_interaction": partial_eta2(ss_interaction),
        "ss_error": ss_error,
    }


if n_seeds < 3:
    print(
        "Phase 2 not available yet (need 3 seeds per cell; "
        f"found {n_seeds}). Run `python run_lifting_ablation.py --phase2` first. "
        "Skipping ANOVA and bootstrap CIs."
    )
else:
    dphgnn_rows = [
        (r["arm"], r["cell_key"], r["test_accuracy"])
        for r in ok_records
        if r["arm"] in ("khop1", "khop2", "knn3") and r["test_accuracy"] is not None
    ]
    try:
        import pandas as pd
        import statsmodels.api as sm
        from statsmodels.formula.api import ols

        df = pd.DataFrame(dphgnn_rows, columns=["arm", "cell", "accuracy"])
        model = ols("accuracy ~ C(arm) * C(cell)", data=df).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        ss_error = anova_table.loc["Residual", "sum_sq"]
        result = {
            "partial_eta2_arm": float(
                anova_table.loc["C(arm)", "sum_sq"]
                / (anova_table.loc["C(arm)", "sum_sq"] + ss_error)
            ),
            "partial_eta2_cell": float(
                anova_table.loc["C(cell)", "sum_sq"]
                / (anova_table.loc["C(cell)", "sum_sq"] + ss_error)
            ),
            "partial_eta2_interaction": float(
                anova_table.loc["C(arm):C(cell)", "sum_sq"]
                / (anova_table.loc["C(arm):C(cell)", "sum_sq"] + ss_error)
            ),
        }
        print(anova_table)
        print("(statsmodels)")
    except ImportError:
        result = manual_two_way_anova(dphgnn_rows)
        print("(statsmodels unavailable — manual sums-of-squares)")

    print()
    print(f"partial eta^2 (lifting arm)      = {result['partial_eta2_arm']:.3f}")
    print(f"partial eta^2 (structural cell)  = {result['partial_eta2_cell']:.3f}")
    print(f"partial eta^2 (arm x cell)       = {result['partial_eta2_interaction']:.3f}")
    print(
        "\nA non-trivial arm x cell interaction is the formal evidence "
        "for H2 (rank inversion): the best lifting is not the same "
        "across structural regimes."
    )

    print("\nBootstrap 95% CI on test accuracy, per arm (pooled over cells):")
    for arm in ["khop1", "khop2", "knn3", "gcn"]:
        vals = [r["test_accuracy"] for r in ok_records if r["arm"] == arm]
        mean, lo, hi = bootstrap_ci(vals)
        print(f"  {ARM_LABELS[arm]:<26s} mean={mean:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]")